# Analítica de Accidentes Laborales OSHA ITA y Clasificación de Riesgos con Modelos de Lenguaje (PLN)
---
Este cuaderno documenta paso a paso el flujo analítico y de modelado:
1. **Ingeniería de Características y Cálculo de Tasas Oficiales OSHA** (TIR y DART normalizadas a 200,000 horas).
2. **Analítica de Accidentabilidad** por Sector Económico (código NAICS) y Ocupación (código SOC).
3. **Modelado de Procesamiento de Lenguaje Natural (PLN)** para clasificar narrativas de accidentes en categorías de Riesgo Crítico:
   - *Modelo Base*: TF-IDF + Regresión Logística.
   - *Modelo Avanzado*: Embeddings Densos con Transformers (Sentence-Transformers ) + Regresión Logística.
4. **Evaluación Comparativa de Desempeño y Métricas de Seguridad Industrial**.

In [1]:
# 1. Importación de librerías y configuración
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
print("Entorno y librerías cargadas con éxito.")

In [2]:
# 2. Carga de los resultados procesados de Tasas de Accidentabilidad OSHA
df_sectores = pd.read_csv("../datos/procesados/tasas_por_sector_naics.csv")
df_ocupaciones = pd.read_csv("../datos/procesados/tasas_por_ocupacion_soc.csv")

print(f"Sectores analizados: {len(df_sectores)}")
df_sectores[["nombre_sector_es", "tir_ponderado", "dart_ponderado", "ltir_ponderado"]].head(10)

In [3]:
# 3. Visualización de Tasas TIR vs DART por Sector Económico
top_sectores = df_sectores.head(8)
x = np.arange(len(top_sectores))
ancho = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bar1 = ax.bar(x - ancho/2, top_sectores["tir_ponderado"], ancho, label="TIR (Tasa de Incidencia Total)", color="#2563EB")
bar2 = ax.bar(x + ancho/2, top_sectores["dart_ponderado"], ancho, label="DART (Casos Severos)", color="#F97316")

ax.set_ylabel("Tasa Oficial por 200,000 Horas Hombre", fontsize=12)
ax.set_title("Top Sectores Industriales con Mayor Tasa de Accidentabilidad OSHA", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(top_sectores["nombre_sector_es"], rotation=35, ha="right")
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

In [4]:
# 4. Evaluación de Métricas Comparativas de Modelos de PLN
with open("../datos/procesados/metricas_comparativas.json", "r", encoding="utf-8") as f:
    metricas = json.load(f)

m_base = metricas["modelo_base_tfidf"]
m_emb = metricas["modelo_avanzado_embeddings"]

tabla_comparativa = pd.DataFrame({
    "Métrica": ["Exactitud (Accuracy)", "F1-Score Macro", "F1-Score Ponderado", "Precisión Macro", "Recall Macro", "Tiempo Inferencia (3,000 casos)"],
    "TF-IDF + Regresión Logística": [f"{m_base['exactitud']*100:.2f}%", f"{m_base['f1_macro']*100:.2f}%", f"{m_base['f1_ponderado']*100:.2f}%", f"{m_base['precision_macro']*100:.2f}%", f"{m_base['recall_macro']*100:.2f}%", f"{m_base['tiempo_inferencia_s']:.3f} s"],
    "Sentence Transformers + Clasificador": [f"{m_emb['exactitud']*100:.2f}%", f"{m_emb['f1_macro']*100:.2f}%", f"{m_emb['f1_ponderado']*100:.2f}%", f"{m_emb['precision_macro']*100:.2f}%", f"{m_emb['recall_macro']*100:.2f}%", f"{m_emb['tiempo_inferencia_s']:.3f} s"]
})
tabla_comparativa